# Person 2 — ML Engineering
Run all cells on the offline synthetic demo, then adapt `config.json`.
Own validation, reproducible baselines, OOF predictions, budgets, blends and the
submission format. Keep fold definitions fixed while comparing hypotheses.
All learned preprocessing and Person 1's feature recipe are fitted inside each fold.

## Challenge update: forecast the full final 199 days

The team reports five-minute prices for 386 stocks in six countries. Forecast prices
throughout the final 199 days from information available before that period; do not
assume later true prices will be revealed. The exact metric and file schema are pending.
The demos below remain generic and do not implement that competition protocol automatically.

Use `qrtkit.forecasting.forecast_prices` for whole-period per-stock persistence or
training-fitted damped drift, and `qrtkit.validation.last_period_holdout` for complete
final-day validation. The [challenge notes](../docs/CHALLENGE_NOTES.md) contain an
adaptable example. Keep future actual prices out of every predictor; use frozen
history for the actual challenge, rather than the optional online financial demo.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "qrtkit").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from qrtkit import load_config, load_data, train_cv
cfg = load_config(ROOT / "config.json")
from copy import deepcopy
from qrtkit.models import available_models
from qrtkit.validation import make_splits
from qrtkit.training import ensemble
from qrtkit.tracking import leaderboard, save_run
from qrtkit.submission import make_submission

# After reviewing Person 1's recipe, set this to ROOT / 'outputs/feature_recipe.json'.
RECIPE_PATH = None
if RECIPE_PATH is not None:
    cfg = load_config(RECIPE_PATH)
data = load_data(cfg)
print('Models available offline:', available_models())
print('Training / test:', data.train.shape, data.test.shape)
print('Task / metric / split:', cfg['task'], cfg['metric'], cfg['validation'])
display(data.train.head())

## Audit the split before training
Use independent row folds only when observations are independent. Group anonymized
dates/subjects; use forward splits for real chronological forecasting. Never infer
chronology from a date label. Temporal gaps and label horizons count distinct timestamps.
An optional `label_end_column` purges labels not available before validation starts.
Temporal OOF coverage below 100% is expected; unvalidated predictions stay missing.

In [ ]:
splits = make_splits(data.train, data.y, cfg)
audit = []
for fold, (tr, va) in enumerate(splits):
    row = {'fold': fold, 'train_rows': len(tr), 'validation_rows': len(va)}
    group = cfg['data'].get('group_column')
    time_col = cfg['data'].get('time_column')
    if group:
        row['group_overlap'] = len(set(data.train.iloc[tr][group]) & set(data.train.iloc[va][group]))
    if time_col:
        row['train_time_max'] = data.train.iloc[tr][time_col].max()
        row['valid_time_min'] = data.train.iloc[va][time_col].min()
    audit.append(row)
display(pd.DataFrame(audit))

## Run the configured baselines
Choose classification `logistic`, `ridge`, `histgb`, `lightgbm`, `xgboost`, `catboost`;
regression also supports `linear`, `lasso`, `elasticnet`, `huber`. Tune tree counts and
thread count for your machine. Ridge classification uses uncalibrated probability
proxies; logistic is a better default for log-loss. Missing optional boosters are
reported; choose an available model explicitly in config.

In [ ]:
results = {}
for name in cfg['models']:
    if not available_models().get(name, False):
        print('Unavailable:', name, '— edit cfg["models"] or install before the event')
        continue
    results[name] = train_cv(data.train, data.y, data.test, cfg, name)
if not results:
    raise RuntimeError('No configured model is available; choose logistic, ridge or histgb')
summary = pd.DataFrame([
    {'model': name, 'oof_score': r.score, 'fold_mean': np.mean(r.fold_scores),
     'fold_std': np.std(r.fold_scores), 'coverage': r.covered.mean(), 'seconds': r.seconds}
    for name, r in results.items()
])
display(summary)
print('Class order:', next(iter(results.values())).classes)

## OOF errors and prediction diversity
Scores retain their natural direction; smaller RMSE/log-loss is better. Compare
only models with the same metric and split hash. Binary OOF probabilities below refer
to `result.classes[1]`. Correlations are diagnostics, not a rule that a blend will improve.

In [ ]:
example = next(iter(results.values()))
covered = example.covered
predictions = pd.DataFrame(index=data.train.index)
for name, r in results.items():
    values = r.oof[:, 1] if cfg['task'] == 'classification' and len(r.classes) == 2 else (
        r.oof.max(axis=1) if cfg['task'] == 'classification' else r.oof)
    predictions[name] = values
display(predictions.loc[covered].corr())
inspection = data.train.loc[covered, cfg['data'].get('id_columns', [])].copy()
inspection['target'] = data.y.loc[covered]
inspection['prediction'] = predictions.loc[covered].iloc[:, 0]
display(inspection.head(15))
display(leaderboard(cfg['output_dir'])[['model', 'metric', 'oof_score', 'coverage', 'split_hash']].tail(10))

## Optional short hyperparameter search
The example budget is 5 seconds / 2 trials so Run All stays quick. Set
`RUN_TUNING=True` and use `cfg['tuning']` values for competition searches if desired.
The limit is soft: a running fit completes, then new folds/trials stop. No complete
trial means increase the budget or reduce folds/trees. Tuning on these same folds
makes the selected CV score optimistic; confirm on an untouched holdout.

In [ ]:
from qrtkit.tuning import tune
RUN_TUNING = True
# Set USE_FULL_BUDGET=True to use the shared tuning settings.
USE_FULL_BUDGET = False
if RUN_TUNING:
    name = cfg['tuning']['model']
    if available_models().get(name, False):
        try:
            tuned = tune(data.train, data.y, cfg, name,
                         seconds=cfg['tuning']['seconds'] if USE_FULL_BUDGET else 5,
                         max_trials=cfg['tuning']['max_trials'] if USE_FULL_BUDGET else 2)
            print('Best parameters:', tuned.best_params, 'selection score:', tuned.best_score)
            display(tuned.trials)
            # Refit inside CV; record the exact tuned parameters in the run artifacts.
            tuned_result = train_cv(data.train, data.y, data.test, cfg, name, params=tuned.best_params)
            results[name + '_tuned'] = tuned_result
        except TimeoutError as exc:
            print(exc)
    else:
        print('Configured tuner model is unavailable; choose one listed above.')

## Fixed-weight ensemble
This example uses equal weights. Preserve the individual models' submissions too.
Changing weights/thresholds to maximize these OOF labels requires another validation
layer to estimate generalization. The API rejects mismatched rows, targets, folds,
class orders and test data. All models are kept in memory; no trained models are saved.

In [ ]:
# Blend the original configured models; keep tuned candidates separate for comparison.
base_results = [results[name] for name in cfg['models'] if name in results]
blend = ensemble(base_results, data.y)
blend.output_path = save_run(blend, cfg, data.train, data.y, data.test)
print('Equal-weight blend:', blend.metric.name, blend.score)
print('Coverage:', blend.covered.mean(), 'fold scores:', blend.fold_scores)
print('OOF initial missing rows:', int((~blend.covered).sum()))

## Generate and inspect submissions
Set `submission.kind` and output columns from the official specification. Binary
probability output uses `positive_class` or class index 1; multiclass output requires
one column per class in `result.classes` order. Label output preserves original labels.
Templates are aligned by identifiers; no-ID templates use explicit positional order.
Generated files stay under ignored output directories. Nothing is uploaded.

In [ ]:
# Inspect each candidate, then choose the appropriate file for the challenge.
for name, r in list(results.items()) + [('equal_weight_blend', blend)]:
    output = Path(r.output_path) / 'submission.csv'
    submission = make_submission(r, data.test, cfg, path=output)
    print(name, submission.shape, output)
    display(submission.head(3))
print('Check IDs, class mapping, row count, metric, and official output specification before submitting.')